# 평가셋(Evaluation DataSet) 구축

### 환경 설정

In [24]:
import os
import sqlite3
import pandas as pd
import chromadb

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain.agents import create_agent
from pathlib import Path
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer


In [3]:
load_dotenv('.env')
load_dotenv('/.env')

if not os.getenv('OPENAI_API_KEY') :
    raise RuntimeError(
        'OpenAI API Key를 확인해 보세요.'
    )
else :
    model = ChatOpenAI(model = 'gpt-4o-mini', temperature=1, timeout=60)
    print('OpenAI API 연결 & MODEL 생성 완료')

OpenAI API 연결 & MODEL 생성 완료


### VectorDB (ChromaDB) 연결 / 임베딩 모델 생성

In [26]:
DB_PATH = Path("../chroma_db")
COLLECTION_NAME = "maplestory_guides"

#_ro_conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True, isolation_level=None, check_same_thread=False)
EMBEDDING_MODEL_NAME = "jhgan/ko-sroberta-multitask"

embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
client = chromadb.PersistentClient(path=str(DB_PATH))
collection = client.get_collection(name=COLLECTION_NAME)

print('데이터베이스 연결 및 임베딩 모델 생성 완료')


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10023.01it/s]


데이터베이스 연결 및 임베딩 모델 생성 완료


### 평가셋 TEST

#### 평가셋 데이터 가져오기

In [29]:
# [rag_eval_questions_100.csv](/C:/Users/Playdata/Desktop/mle-01-p1-team3/docs/rag_eval_questions_100.csv)

evalset = pd.read_csv('../docs/rag_eval_gold_100.csv')
print('페이지 수 : ', len(evalset))

display(evalset.head(3))


페이지 수 :  100


,eval_id,persona,category,question,question_variant,source_type,difficulty,answerable,gold_source_refs,gold_source_ids,gold_source_ref_ids,gold_source_urls,reference_answer,must_include,nice_to_include,must_not_include,scoring_focus
0,E001,초보,게임시작,메이플스토리를 처음 시작하려면 무엇부터 해야 하나요?,처음 설치한 뒤 어떤 순서로 시작하면 되나요?,guide,easy,yes,게임 시작,chunk_0; chunk_1; chunk_2; chunk_3; chunk_4; c...,article:272,https://maplestory.nexon.com/Guide/N23GameInfo...,"공식 가이드 기준으로 시작 절차를 순서대로 설명하고, 초보가 바로 따라할 수 있게 ...",단계적 시작 절차; 초보 기준 설명; 관련 핵심 기능 또는 준비물; 바로 실행 가능...,초보가 바로 실행할 행동; 용어를 쉬운 말로 풀어 설명,문서에 없는 최신 정보 단정; 근거 없는 과장; 초보 맥락 없는 고인물식 설명,"근거 충실성, 단계적 설명, 실행 가능성"
1,E002,초보,게임시작,메이플스토리는 어떻게 설치하고 실행하나요?,게임을 시작하려면 설치 절차가 어떻게 되나요?,guide,easy,yes,게임 시작,chunk_0; chunk_1; chunk_2; chunk_3; chunk_4; c...,article:272,https://maplestory.nexon.com/Guide/N23GameInfo...,"공식 가이드 기준으로 시작 절차를 순서대로 설명하고, 초보가 바로 따라할 수 있게 ...",단계적 시작 절차; 초보 기준 설명; 관련 핵심 기능 또는 준비물; 바로 실행 가능...,초보가 바로 실행할 행동; 용어를 쉬운 말로 풀어 설명,문서에 없는 최신 정보 단정; 근거 없는 과장; 초보 맥락 없는 고인물식 설명,"근거 충실성, 단계적 설명, 실행 가능성"
2,E003,초보,게임시작,처음 시작할 때 선택할 수 있는 직업은 어떤 종류가 있나요?,초반에 어떤 직업군을 고를 수 있나요?,guide,easy,yes,메이플스토리 신규 · 복귀 용사 가이드,chunk_1378; chunk_1379; chunk_1380; chunk_1381...,article:147377,https://maplestory.nexon.com/Guide/N23GameInfo...,"공식 가이드 기준으로 시작 절차를 순서대로 설명하고, 초보가 바로 따라할 수 있게 ...",단계적 시작 절차; 초보 기준 설명; 관련 핵심 기능 또는 준비물; 바로 실행 가능...,초보가 바로 실행할 행동; 용어를 쉬운 말로 풀어 설명,문서에 없는 최신 정보 단정; 근거 없는 과장; 초보 맥락 없는 고인물식 설명,"근거 충실성, 단계적 설명, 실행 가능성"


##### 함수 정의

In [57]:
def search_ids(question: str, k: int = 3):
    """질문과 가장 가까운 조각 k개의 id를 순위 순서로 돌려준다."""
    query_embedding = embed_model.encode(
        [question],
        normalize_embeddings = True
    )

    result = collection.query(
        query_embeddings = query_embedding.tolist(),
        n_results = k,
        include = []
    )

    return result["ids"][0]

In [72]:
# 평가셋(evalset)구성
# 'eval_id', 'persona', 'category', 'question', 
# 'question_variant', 'source_type', 'difficulty', 
# 'answerable', 'gold_source_refs', 'gold_source_ids', 
# 'gold_source_ref_ids', 'gold_source_urls', 'reference_answer', 
# 'must_include', 'nice_to_include', 'must_not_include', 
# 'scoring_focus'

# 가장 무난한 질문 선택 (또는 평가셋 번호, 예: 'E001' ~ 'E100')
credit_eval_id = "E090"

# 1. 평가셋에서 문항 찾기
credit_row = evalset[evalset["eval_id"] == credit_eval_id].iloc[0]
credit_query = credit_row["question"]

# 2. 실제 질문으로 검색
top3 = search_ids(credit_query, k=3)

for rank, chunk_id in enumerate(top3, 1):
    print(f"{rank}위 {chunk_id}")

# 3. gold chunk_id 정리
credit_gold = [x.strip() for x in credit_row["gold_source_ids"].split(";") if x.strip()]

print(f"평가셋 문항 {credit_row['eval_id']}의 정답 조각 : {credit_gold}")

print("=" * 100)

# 4. chunk 단위 평가
hit_k = int(bool(set(top3) & set(credit_gold)))
recall_k = len(set(top3) & set(credit_gold)) / len(set(credit_gold))
precision_k = len(set(top3) & set(credit_gold)) / len(top3)

print("Hit@K 평가 :", hit_k)
print("Recall@K 평가 :", recall_k)
print("Precision@K 평가 :", precision_k)

1위 chunk_1380
2위 chunk_8
3위 chunk_1390
평가셋 문항 E090의 정답 조각 : ['chunk_1378', 'chunk_1379', 'chunk_1380', 'chunk_1381', 'chunk_1382', 'chunk_1383', 'chunk_1384', 'chunk_1385', 'chunk_1386', 'chunk_1387', 'chunk_1388', 'chunk_1389', 'chunk_1390', 'chunk_1391', 'chunk_1392', 'chunk_1393', 'chunk_1394', 'chunk_1395', 'chunk_1396', 'chunk_1397', 'chunk_1398', 'chunk_1399', 'chunk_1400', 'chunk_1401', 'chunk_1402', 'chunk_1403', 'chunk_1404', 'chunk_1405', 'chunk_1406', 'chunk_1407', 'chunk_1408', 'chunk_1409', 'chunk_1410', 'chunk_1411', 'chunk_1412', 'chunk_1413', 'chunk_1414', 'chunk_1415', 'chunk_1416', 'chunk_1417', 'chunk_1418', 'chunk_1419', 'chunk_1420', 'chunk_1421', 'chunk_1422', 'chunk_1423', 'chunk_1424', 'chunk_1425', 'chunk_1426', 'chunk_1427', 'chunk_1428', 'chunk_1429', 'chunk_1430', 'chunk_1431', 'chunk_1432', 'chunk_1433', 'chunk_1434', 'chunk_1435', 'chunk_1436', 'chunk_1437', 'chunk_1438', 'chunk_1439', 'chunk_1440', 'chunk_1441', 'chunk_1442', 'chunk_1443', 'chunk_1444', '

In [75]:
print('질문 : ', credit_query)
print()

for rank in top3 :
    result = collection.get(ids=rank, include=["documents", "metadatas"])
    print(result["documents"][0])
    print(result["metadatas"][0])
    print()

for gold in credit_gold :
    result = collection.get(ids=gold, include=["documents", "metadatas"])
    print(result["documents"][0])
    print(result["metadatas"][0])
    print()


질문 :  초보에게 무난한 직업을 고를 때 어떤 기준으로 보면 좋을까요?

클릭하면 자세한 내용을 확인하실 수 있습니다 START 게임 시작하기 Q메이플스토리는 어떻게 시작하나요 Q어떤 직업을 선택할 수 있나요 Q처음 시작하면 무엇부터 해야 하나요 RETURN 복귀 용사님을 위한 안내 Q복귀하면 기존 캐릭터를 이어서 키울 수 있나요 Q오랜만에 메이플스토리에 돌아왔는데 바뀐 시스템에 잘 적응할 수 있을까요 GROWTH 캐릭터 성장
{'section_title': '기타/TIP', 'name': '메이플스토리 신규 · 복귀 용사 가이드', 'source': 'guide', 'board_id': 429467345, 'chunk_index': 1380, 'url': 'https://maplestory.nexon.com/Guide/N23GameInformation/Articles/147377', 'article_id': 147377}

선택하기 마음에 드는 직업을 선택해 플레이할 수 있습니다 단 생성이 제한된 캐릭터는 생성이 불가능합니다 각 캐릭터에 대한 자세한 사항은 아래 링크를 통해 확인할 수 있습니다 직업소개 바로가기 2 캐릭터 설정하기 캐릭터 이름 설정 시 같은 캐릭터명을 가진 캐릭터가 있거나 금칙어가 포함될 경우 이름으로 설정이 불가능합니다 이름을 정했다면 원하는 캐릭터 성별
{'chunk_index': 8, 'section_title': '기초 가이드', 'article_id': 373, 'source': 'guide', 'board_id': 429467337, 'url': 'https://maplestory.nexon.com/Guide/N23GameInformation/Articles/373', 'name': '캐릭터 생성/삭제'}

Q-어떤 직업을 선택할 수 있나요? A-메이플스토리에는 전사 · 마법사 · 궁수 · 도적 · 해적의 다섯 계열로 구분되는 여러 다양한 직업이 존재하며, 출신에 따라 모험가 · 시그너스 기사단 · 영웅 · 레지스탕스 등의 직업군